# Banana Doctor Two-Stage Training

Notebook ini melatih pipeline:
1. **Banana Gate**: `Banana Leaf` vs `Not Banana Leaf`.
2. **Disease Ensemble**: klasifikasi 8 kelas daun pisang hanya jika gate lolos.

Input dataset: `/content/drive/MyDrive/banana-datasets/banana_datasets_twostage.zip`.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

## 2. Extract Dataset

In [ ]:
import os
import json
import shutil
from pathlib import Path

DATASET_ZIP = Path('/content/drive/MyDrive/banana-datasets/banana_datasets_twostage.zip')
RAW_DATA_DIR = Path('/content/datasets')
WORK_DIR = Path('/content/twostage')
GATE_DATA_DIR = WORK_DIR / 'banana_gate_dataset'
DISEASE_DATA_DIR = WORK_DIR / 'disease_dataset'
OUTPUT_DIR = Path('/content/artifacts')

assert DATASET_ZIP.exists(), f'Dataset zip tidak ditemukan: {DATASET_ZIP}'

for p in [RAW_DATA_DIR, WORK_DIR, OUTPUT_DIR]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

LOCAL_ZIP = Path('/content/banana_datasets_twostage.zip')
shutil.copy2(DATASET_ZIP, LOCAL_ZIP)
!unzip -q "{LOCAL_ZIP}" -d "{RAW_DATA_DIR}"

print('Isi dataset mentah:')
for d in sorted(p for p in RAW_DATA_DIR.iterdir() if p.is_dir()):
    n = sum(1 for root, _, files in os.walk(d) for f in files)
    print(f'  {d.name}: {n}')

## 3. Imports dan Konfigurasi

In [ ]:
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import MobileNetV2, ResNet50, InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
GATE_HEAD_EPOCHS = 20
GATE_FINE_TUNE_EPOCHS = 30
DISEASE_EPOCHS = 80
LABEL_SMOOTHING = 0.05

BANANA_GATE_LABEL = 'Banana Leaf'
NEG_LABEL = 'Not Banana Leaf'
BANANA_LABELS = [
    'Augmented Banana Black Sigatoka Disease',
    'Augmented Banana Bract Mosaic Virus Disease',
    'Augmented Banana Cordana Disease',
    'Augmented Banana Healthy Leaf',
    'Augmented Banana Insect Pest Disease',
    'Augmented Banana Moko Disease',
    'Augmented Banana Panama Disease',
    'Augmented Banana Yellow Sigatoka Disease',
]

DEFAULT_GATE_THRESHOLD = 0.80
TARGET_NOT_BANANA_RECALL = 0.98
TARGET_BANANA_RECALL = 0.95

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 4. Susun Dataset Gate dan Disease

In [ ]:
def list_images(root):
    root = Path(root)
    return [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXT and p.is_file()]

def copy_images(src_dir, dst_dir, prefix):
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for src in list_images(src_dir):
        out = dst_dir / f'{prefix}_{count:06d}{src.suffix.lower()}'
        shutil.copy2(src, out)
        count += 1
    return count

for p in [GATE_DATA_DIR, DISEASE_DATA_DIR]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

# Stage 1: binary gate dataset
gate_banana_dir = GATE_DATA_DIR / BANANA_GATE_LABEL
gate_neg_dir = GATE_DATA_DIR / NEG_LABEL

banana_total = 0
for label in BANANA_LABELS:
    src = RAW_DATA_DIR / label
    assert src.exists(), f'Folder pisang tidak ditemukan: {src}'
    banana_total += copy_images(src, gate_banana_dir, label.lower().replace(' ', '')[:14])
    shutil.copytree(src, DISEASE_DATA_DIR / label, dirs_exist_ok=True)

neg_src = RAW_DATA_DIR / NEG_LABEL
assert neg_src.exists(), f'Folder negatif tidak ditemukan: {neg_src}'
neg_total = copy_images(neg_src, gate_neg_dir, 'notbanana')

print('Gate dataset:')
print(f'  {BANANA_GATE_LABEL}: {banana_total}')
print(f'  {NEG_LABEL}: {neg_total}')

print('\nDisease dataset:')
for label in BANANA_LABELS:
    print(f'  {label}: {len(list_images(DISEASE_DATA_DIR / label))}')

## 5. Data Generator Gate

In [ ]:
gate_train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.12,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=(0.75, 1.25),
    fill_mode='nearest',
)

gate_val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
gate_classes = [NEG_LABEL, BANANA_GATE_LABEL]

gate_train = gate_train_datagen.flow_from_directory(
    GATE_DATA_DIR,
    classes=gate_classes,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=SEED,
)

gate_val = gate_val_datagen.flow_from_directory(
    GATE_DATA_DIR,
    classes=gate_classes,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
)

print('Gate class indices:', gate_train.class_indices)
assert gate_train.class_indices[BANANA_GATE_LABEL] == 1, gate_train.class_indices

## 6. Train Banana Gate dengan Fine-Tuning

In [ ]:
def gate_callbacks(prefix):
    return [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        ModelCheckpoint(str(OUTPUT_DIR / 'banana_gate.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    ]

def build_gate_model():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base.trainable = False
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dropout(0.35),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.35),
        Dense(1, activation='sigmoid'),
    ])
    return model, base

gate_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(gate_train.classes),
    y=gate_train.classes,
)
gate_class_weight = {i: float(w) for i, w in enumerate(gate_weights_array)}
print('Gate class weights:', gate_class_weight)

banana_gate, gate_base = build_gate_model()
banana_gate.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')],
)

history_gate_head = banana_gate.fit(
    gate_train,
    epochs=GATE_HEAD_EPOCHS,
    validation_data=gate_val,
    class_weight=gate_class_weight,
    callbacks=gate_callbacks('head'),
    verbose=1,
)

# Fine-tune layer atas MobileNetV2, BatchNorm tetap frozen agar stabil.
gate_base.trainable = True
for layer in gate_base.layers[:-30]:
    layer.trainable = False
for layer in gate_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

banana_gate.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')],
)

history_gate_ft = banana_gate.fit(
    gate_train,
    epochs=GATE_HEAD_EPOCHS + GATE_FINE_TUNE_EPOCHS,
    initial_epoch=len(history_gate_head.history['loss']),
    validation_data=gate_val,
    class_weight=gate_class_weight,
    callbacks=gate_callbacks('fine_tune'),
    verbose=1,
)

banana_gate.save(OUTPUT_DIR / 'banana_gate.keras')

## 7. Pilih Threshold Gate dari Validation Set

In [ ]:
def choose_gate_threshold(y_true, banana_prob):
    thresholds = np.round(np.arange(0.50, 0.991, 0.01), 2)
    rows = []
    for th in thresholds:
        pred_banana = banana_prob >= th
        true_banana = y_true == 1
        true_not = y_true == 0
        banana_recall = float(np.sum(pred_banana & true_banana) / max(1, np.sum(true_banana)))
        not_banana_recall = float(np.sum((~pred_banana) & true_not) / max(1, np.sum(true_not)))
        balanced = (banana_recall + not_banana_recall) / 2
        rows.append({
            'threshold': float(th),
            'banana_recall': banana_recall,
            'not_banana_recall': not_banana_recall,
            'balanced_accuracy': balanced,
        })

    feasible = [
        r for r in rows
        if r['not_banana_recall'] >= TARGET_NOT_BANANA_RECALL
        and r['banana_recall'] >= TARGET_BANANA_RECALL
    ]
    if feasible:
        chosen = max(feasible, key=lambda r: (r['not_banana_recall'], r['banana_recall'], r['balanced_accuracy'], -r['threshold']))
        warning = None
    else:
        chosen = max(rows, key=lambda r: (2 * r['not_banana_recall'] + r['banana_recall'], r['balanced_accuracy'], -abs(r['threshold'] - DEFAULT_GATE_THRESHOLD)))
        warning = 'Target recall tidak tercapai; threshold terbaik berbasis skor prioritas dipakai.'
    return chosen, rows, warning

gate_val.reset()
gate_probs = banana_gate.predict(gate_val, verbose=0).reshape(-1)
y_gate_true = gate_val.classes.astype(int)
chosen_threshold, threshold_table, threshold_warning = choose_gate_threshold(y_gate_true, gate_probs)
banana_threshold = chosen_threshold['threshold']

gate_pred = (gate_probs >= banana_threshold).astype(int)
gate_report = classification_report(y_gate_true, gate_pred, target_names=[NEG_LABEL, BANANA_GATE_LABEL])
gate_cm = confusion_matrix(y_gate_true, gate_pred)

print('Chosen gate threshold:', banana_threshold)
print(json.dumps(chosen_threshold, indent=2))
if threshold_warning:
    print('WARNING:', threshold_warning)
print(gate_report)

with open(OUTPUT_DIR / 'banana_gate_report.txt', 'w') as f:
    f.write('Banana Gate Classification Report\n')
    f.write('=' * 60 + '\n\n')
    f.write(gate_report)
    f.write('\n\nChosen threshold:\n')
    f.write(json.dumps(chosen_threshold, indent=2))
    if threshold_warning:
        f.write('\n\nWARNING: ' + threshold_warning)

plt.figure(figsize=(6, 5))
sns.heatmap(gate_cm, annot=True, fmt='d', cmap='Greens', xticklabels=[NEG_LABEL, BANANA_GATE_LABEL], yticklabels=[NEG_LABEL, BANANA_GATE_LABEL])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Banana Gate Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_banana_gate.png', dpi=150)
plt.show()

banana_gate_config = {
    'pipeline': 'two_stage',
    'model': 'banana_gate.keras',
    'input_size': IMAGE_SIZE[0],
    'labels': [NEG_LABEL, BANANA_GATE_LABEL],
    'banana_label': BANANA_GATE_LABEL,
    'banana_index': 1,
    'banana_threshold': banana_threshold,
    'default_threshold': DEFAULT_GATE_THRESHOLD,
    'target_not_banana_recall': TARGET_NOT_BANANA_RECALL,
    'target_banana_recall': TARGET_BANANA_RECALL,
    'validation_metrics': chosen_threshold,
    'warning': threshold_warning,
}
with open(OUTPUT_DIR / 'banana_gate_config.json', 'w') as f:
    json.dump(banana_gate_config, f, indent=2)

## 8. Disease Data Generator

In [ ]:
disease_train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=(0.80, 1.20),
    fill_mode='nearest',
)

disease_val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = disease_train_datagen.flow_from_directory(
    DISEASE_DATA_DIR,
    classes=BANANA_LABELS,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED,
)

val_generator = disease_val_datagen.flow_from_directory(
    DISEASE_DATA_DIR,
    classes=BANANA_LABELS,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

labels = list(train_generator.class_indices.keys())
NUM_CLASSES = len(labels)
assert labels == BANANA_LABELS, labels
assert NUM_CLASSES == 8, NUM_CLASSES

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes,
)
class_weights_dict = {i: float(class_weights[i]) for i in range(len(class_weights))}
print(f'Kelas penyakit ({NUM_CLASSES}): {labels}')
print('Class weights:', {labels[i]: round(w, 3) for i, w in class_weights_dict.items()})

## 9. Helper Disease Models

In [ ]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        ModelCheckpoint(filepath=str(OUTPUT_DIR / f'best_{model_name}.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4, min_lr=1e-7, verbose=1),
    ]

def plot_history(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'{model_name} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'{model_name} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'history_{model_name}.png', dpi=150)
    plt.show()

def evaluate_model(model, generator, model_name):
    generator.reset()
    y_true = generator.classes
    y_pred = np.argmax(model.predict(generator, verbose=0), axis=1)
    print(f'\nEvaluasi: {model_name}')
    print(classification_report(y_true, y_pred, target_names=labels))
    return y_pred

def build_cnn_model():
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)),
        BatchNormalization(), Conv2D(32, (3, 3), activation='relu'), MaxPooling2D((2, 2)), Dropout(0.25),
        Conv2D(64, (3, 3), activation='relu'), BatchNormalization(), Conv2D(64, (3, 3), activation='relu'), MaxPooling2D((2, 2)), Dropout(0.25),
        Conv2D(128, (3, 3), activation='relu'), BatchNormalization(), Conv2D(128, (3, 3), activation='relu'), MaxPooling2D((2, 2)), Dropout(0.25),
        Conv2D(256, (3, 3), activation='relu'), BatchNormalization(), Conv2D(256, (3, 3), activation='relu'), GlobalAveragePooling2D(), Dropout(0.5),
        Dense(256, activation='relu'), BatchNormalization(), Dropout(0.5),
        Dense(128, activation='relu'), BatchNormalization(), Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])

def build_resnet_model():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base.trainable = False
    return Sequential([base, GlobalAveragePooling2D(), Dense(128, activation='relu'), BatchNormalization(), Dropout(0.5), Dense(NUM_CLASSES, activation='softmax')])

def build_inception_model():
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base.trainable = False
    return Sequential([base, GlobalAveragePooling2D(), Dense(128, activation='relu'), BatchNormalization(), Dropout(0.5), Dense(NUM_CLASSES, activation='softmax')])

## 10. Train Disease Ensemble

In [ ]:
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

model_cnn = build_cnn_model()
model_cnn.compile(optimizer=Adam(learning_rate=0.001), loss=loss_fn, metrics=['accuracy'])
history_cnn = model_cnn.fit(train_generator, epochs=DISEASE_EPOCHS, validation_data=val_generator, class_weight=class_weights_dict, callbacks=get_callbacks('cnn'), verbose=1)
plot_history(history_cnn, 'Custom CNN')

model_resnet = build_resnet_model()
model_resnet.compile(optimizer=Adam(learning_rate=0.001), loss=loss_fn, metrics=['accuracy'])
history_resnet = model_resnet.fit(train_generator, epochs=DISEASE_EPOCHS, validation_data=val_generator, class_weight=class_weights_dict, callbacks=get_callbacks('resnet'), verbose=1)
plot_history(history_resnet, 'ResNet50')

model_inception = build_inception_model()
model_inception.compile(optimizer=Adam(learning_rate=0.001), loss=loss_fn, metrics=['accuracy'])
history_inception = model_inception.fit(train_generator, epochs=DISEASE_EPOCHS, validation_data=val_generator, class_weight=class_weights_dict, callbacks=get_callbacks('inception'), verbose=1)
plot_history(history_inception, 'InceptionV3')

## 11. Evaluasi dan Simpan Artifacts

In [ ]:
pred_cnn = evaluate_model(model_cnn, val_generator, 'Custom CNN')
pred_resnet = evaluate_model(model_resnet, val_generator, 'ResNet50')
pred_inception = evaluate_model(model_inception, val_generator, 'InceptionV3')

val_generator.reset()
prob_cnn = model_cnn.predict(val_generator, verbose=0)
val_generator.reset()
prob_resnet = model_resnet.predict(val_generator, verbose=0)
val_generator.reset()
prob_inception = model_inception.predict(val_generator, verbose=0)

ensemble_prob = (prob_cnn + prob_resnet + prob_inception) / 3
ensemble_pred = np.argmax(ensemble_prob, axis=1)
y_true = val_generator.classes

acc_cnn = accuracy_score(y_true, pred_cnn)
acc_resnet = accuracy_score(y_true, pred_resnet)
acc_inception = accuracy_score(y_true, pred_inception)
acc_ensemble = accuracy_score(y_true, ensemble_pred)

print('\nClassification Report - Disease Ensemble:')
report = classification_report(y_true, ensemble_pred, target_names=labels)
print(report)

cm = confusion_matrix(y_true, ensemble_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Disease Ensemble')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_ensemble.png', dpi=150)
plt.show()

model_cnn.save(OUTPUT_DIR / 'model_cnn.keras')
model_resnet.save(OUTPUT_DIR / 'model_resnet.keras')
model_inception.save(OUTPUT_DIR / 'model_inception.keras')

with open(OUTPUT_DIR / 'labels.json', 'w') as f:
    json.dump(labels, f, indent=2)

ensemble_config = {
    'pipeline': 'two_stage',
    'banana_gate': {
        'config': 'banana_gate_config.json',
        'model': 'banana_gate.keras',
        'banana_threshold': banana_threshold,
    },
    'models': [
        {'name': 'Custom CNN', 'file': 'model_cnn.keras'},
        {'name': 'ResNet50', 'file': 'model_resnet.keras'},
        {'name': 'InceptionV3', 'file': 'model_inception.keras'},
    ],
    'input_size': IMAGE_SIZE[0],
    'num_classes': len(labels),
    'labels': labels,
    'voting': 'soft',
    'accuracy': {
        'cnn': float(acc_cnn),
        'resnet': float(acc_resnet),
        'inception': float(acc_inception),
        'ensemble': float(acc_ensemble),
    },
}
with open(OUTPUT_DIR / 'ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

with open(OUTPUT_DIR / 'evaluation_report.txt', 'w') as f:
    f.write('Disease Ensemble Classification Report\n')
    f.write('=' * 60 + '\n\n')
    f.write(report)

print('\nSemua artifact tersimpan di:', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size
        print(f'  {f.name} ({size/1024/1024:.1f} MB)' if size > 1024*1024 else f'  {f.name} ({size/1024:.1f} KB)')

## 12. Copy Artifacts ke Drive

In [ ]:
DRIVE_OUTPUT = Path('/content/drive/MyDrive/banana-disease-artifacts')
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
print(f'Artifacts copied to Google Drive: {DRIVE_OUTPUT}')

shutil.make_archive('/content/artifacts_download', 'zip', OUTPUT_DIR)
print('Zip artifacts tersedia di: /content/artifacts_download.zip')